# Foundations — ETL Walkthrough

Runs `etl_pipeline.py`'s actual functions (imported, not duplicated) one step at a time, so each transformation and each data-quality fix can be shown with real before/after output — easier to demo than reading the `.py` file top to bottom.

Covers Task 1.1 (`projects.csv`) and Task 1.3 (`employees.csv`).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # so `import etl_pipeline` finds the .py file one level up (this notebook lives in notebooks/)

import pandas as pd
from etl_pipeline import (
    load_projects, transform_projects,
    load_employees, clean_employees,
    DATA_DIR,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

## Task 1.1 — `projects.csv`

### Step 1 — Load the raw file

The `INFO` lines below are the pipeline's own logging (from `load_projects()`), not notebook output — they just confirm the file loaded. The actual preview is in the next cell.

In [2]:
raw_projects = load_projects(os.path.join(DATA_DIR, "projects.csv"))

2026-09-24 18:09:56,776 | INFO | Loading projects data from C:\Users\anil.vaikuntam\OneDrive - G42\Documents\Stack Up Enginerring\stackup-engineering-academy_assessment\datasets\projects.csv


2026-09-24 18:09:56,794 | INFO | Loaded 500 project rows, 11 columns


In [3]:
print("Shape (rows, columns):", raw_projects.shape)
raw_projects.head()

Shape (rows, columns): (500, 11)


,project_id,project_name,department,status,start_date,end_date,budget,actual_cost,project_manager_id,priority,region
0,PRJ0001,MLOps Platform,Sustainability,Completed,2023-06-13,2023-12-01,1200000.0,907809.0,EMP0126,Medium,Dubai
1,PRJ0002,Tax Automation Platform - Q2,Marketing,In Progress,2025-02-28,NaT,800000.0,795986.0,EMP0045,Medium,Abu Dhabi
2,PRJ0003,ERP Consolidation Phase 2,Legal,Completed,2023-11-21,2024-12-16,80000.0,56981.0,EMP0405,Medium,Dubai
3,PRJ0004,Employee Wellness App - Q1,Marketing,Completed,2024-06-09,2025-08-17,800000.0,707314.0,EMP0270,Medium,Abu Dhabi
4,PRJ0005,Mobile Workforce App - Q4,Finance,In Progress,2023-04-25,NaT,2000000.0,1329426.0,EMP0478,High,Ras Al Khaimah


### Step 2 — What's missing before we touch anything

`budget` and `actual_cost` both have nulls — this is why they get filled with 0 *before* any derived column is calculated, not after.

In [4]:
raw_projects[["budget", "actual_cost"]].isna().sum()

budget         32
actual_cost    60
dtype: int64

### Step 3 — Transform: fill nulls, add 6 derived columns

In [5]:
projects_clean = transform_projects(raw_projects)
print(projects_clean.shape)
projects_clean[["project_id", "budget", "actual_cost", "budget_variance",
                "is_over_budget", "duration_days", "budget_utilisation_pct",
                "status_category", "risk_level"]].head(10)

2026-09-24 18:09:56,849 | INFO | Transforming projects data...


2026-09-24 18:09:56,851 | INFO | Null budget: 32 rows | Null actual_cost: 60 rows -> filled with 0


2026-09-24 18:09:56,868 | INFO | Transform complete: 500 rows, 17 columns


2026-09-24 18:09:56,871 | INFO | risk_level distribution: {'High': 188, 'Low': 167, 'Medium': 145}


(500, 17)


,project_id,budget,actual_cost,budget_variance,is_over_budget,duration_days,budget_utilisation_pct,status_category,risk_level
0,PRJ0001,1200000.0,907809.0,-292191.0,False,171.0,75.650750,Closed,Low
1,PRJ0002,800000.0,795986.0,-4014.0,False,NaN,99.498250,Active,Medium
2,PRJ0003,80000.0,56981.0,-23019.0,False,391.0,71.226250,Closed,Low
3,PRJ0004,800000.0,707314.0,-92686.0,False,434.0,88.414250,Closed,Low
4,PRJ0005,2000000.0,1329426.0,-670574.0,False,NaN,66.471300,Active,Medium
5,PRJ0006,2000000.0,1438154.0,-561846.0,False,NaN,71.907700,Active,Medium
6,PRJ0007,500000.0,437614.0,-62386.0,False,470.0,87.522800,Closed,Low
7,PRJ0008,200000.0,172385.0,-27615.0,False,NaN,86.192500,Active,Low
8,PRJ0009,1200000.0,1115876.0,-84124.0,False,NaN,92.989667,Active,Medium
9,PRJ0010,800000.0,541681.0,-258319.0,False,NaN,67.710125,Active,High


### Step 4 — Sanity check: is `risk_level` skewed to one bucket?

In [6]:
projects_clean["risk_level"].value_counts()

risk_level
High      188
Low       167
Medium    145
Name: count, dtype: int64

## Task 1.3 — `employees.csv`

### Step 1 — Load the raw file

Same as before: the `INFO` lines below are the pipeline's own logging (from `load_employees()`) — the null-count check is a quick sanity pass, not the real data-quality detection (that's `clean_employees()`, next).

In [7]:
raw_employees = load_employees(os.path.join(DATA_DIR, "employees.csv"))

2026-09-24 18:09:56,902 | INFO | Loading employees data from C:\Users\anil.vaikuntam\OneDrive - G42\Documents\Stack Up Enginerring\stackup-engineering-academy_assessment\datasets\employees.csv


2026-09-24 18:09:56,915 | INFO | Loaded 1000 employee rows, 12 columns


2026-09-24 18:09:56,917 | INFO | Null counts per column:
email    10


In [8]:
print("Shape (rows, columns):", raw_employees.shape)
raw_employees.head()

Shape (rows, columns): (1000, 12)


,employee_id,full_name,email,department,role,level,hire_date,salary,manager_id,region,status,years_experience
0,EMP0001,Anjali Desai,anjali.desai@presight.ai,Operations,Senior Data Engineer,Mid,2019-08-06,17839,EMP0000,Abu Dhabi,Active,6
1,EMP0002,Mohammed Mansour,mohammed.mansour@presight.ai,Operations,Senior Data Engineer,Senior,2014-08-20,33928,EMP0000,Dubai,Active,11
2,EMP0003,Adam Rahman,adam.rahman@presight.ai,Legal,Compliance Analyst,Mid,2021-02-17,18763,EMP0000,Abu Dhabi,Active,4
3,EMP0004,Fatima Al Suwaidi,fatima.al.suwaidi@presight.ai,Procurement,Procurement Analyst,Mid,2024-05-30,20100,EMP0000,Dubai,Active,2
4,EMP0005,Priya Chen,priya.chen@presight.ai,Procurement,Procurement Analyst,Junior,2024-02-21,14370,EMP0000,Abu Dhabi,Active,0


### Step 2 — Run the full cleaning pass

`clean_employees()` finds and fixes all 6 issue categories in one pass. The cells below re-derive each issue's condition (same logic as in `etl_pipeline.py`) just to show real before/after rows for each one.

In [9]:
employees_clean = clean_employees(raw_employees)
qs = employees_clean.attrs["quality_summary"]

summary = pd.DataFrame([
    ["1", "Blank email",                    qs["missing_email_fixed"],              "Placeholder address"],
    ["2", "hire_date not a valid date",      qs["invalid_hire_date_format_fixed"],   "Set to null"],
    ["3", "hire_date outside 1990–today",     qs["implausible_hire_date_fixed"],      "Set to null"],
    ["4", "years_experience outside 0–50",   qs["years_experience_out_of_range_fixed"], "Set to level median"],
    ["5", "Salary outside level's band",     qs["salary_level_mismatch_fixed"],      "Capped to level p95"],
    ["6", "Active under Inactive manager",   qs["status_conflicts_found"],           "Guardrail (0 found)"],
], columns=["Issue #", "Issue", "Rows found", "Fix applied"])

print(f"Total rows fixed: {sum(qs.values())} across {len(raw_employees)} employees")
summary

2026-09-24 18:09:56,939 | INFO | Cleaning employees data...


2026-09-24 18:09:56,941 | INFO | Issue 1 [Missing values] — blank emails: 10 rows


2026-09-24 18:09:56,943 | INFO | Issue 2 [Invalid date format] — non-date-shaped hire_date: 5 rows


2026-09-24 18:09:56,947 | INFO | Issue 3 [Implausible dates] — hire_date outside 1990-today: 3 rows


2026-09-24 18:09:56,950 | INFO | Issue 4 [Numeric out-of-range] — years_experience outside [0,50]: 5 rows


2026-09-24 18:09:56,971 | INFO | Issue 5 [Logical inconsistency] — salary inconsistent with level band: 3 rows


2026-09-24 18:09:56,979 | INFO | Issue 6 [Status conflict] — Active employee reporting to an Inactive manager: 0 rows


2026-09-24 18:09:56,980 | INFO | Data quality summary: {'missing_email_fixed': 10, 'invalid_hire_date_format_fixed': 5, 'implausible_hire_date_fixed': 3, 'years_experience_out_of_range_fixed': 5, 'salary_level_mismatch_fixed': 3, 'status_conflicts_found': 0}


Total rows fixed: 26 across 1000 employees


,Issue #,Issue,Rows found,Fix applied
0,1,Blank email,10,Placeholder address
1,2,hire_date not a valid date,5,Set to null
2,3,hire_date outside 1990–today,3,Set to null
3,4,years_experience outside 0–50,5,Set to level median
4,5,Salary outside level's band,3,Capped to level p95
5,6,Active under Inactive manager,0,Guardrail (0 found)


### Issue 1 — Blank emails

10 rows. Fixed with a placeholder address, not a guessed one.

In [10]:
missing_email = raw_employees["email"].isna() | (raw_employees["email"].astype(str).str.strip() == "")
print("rows affected:", missing_email.sum())

before = raw_employees.loc[missing_email, ["employee_id", "email"]].head()
after = employees_clean.loc[missing_email, ["employee_id", "email"]].head()
before.merge(after, on="employee_id", suffixes=("_before", "_after"))

rows affected: 10


,employee_id,email_before,email_after
0,EMP0358,NaN,unknown@presight.ai
1,EMP0373,NaN,unknown@presight.ai
2,EMP0461,NaN,unknown@presight.ai
3,EMP0536,NaN,unknown@presight.ai
4,EMP0647,NaN,unknown@presight.ai


### Issue 2 & 3 — Broken / implausible hire dates

8 rows total (5 non-date-shaped + 3 outside 1990–today). Set to null rather than guessed.

In [11]:
bad_hire_date = raw_employees["employee_id"].isin(
    employees_clean.loc[employees_clean["hire_date"].isna() & raw_employees["hire_date"].notna(), "employee_id"]
)
before = raw_employees.loc[bad_hire_date, ["employee_id", "hire_date"]]
after = employees_clean.loc[bad_hire_date, ["employee_id", "hire_date"]]
before.merge(after, on="employee_id", suffixes=("_before", "_after"))

,employee_id,hire_date_before,hire_date_after
0,EMP0038,-999,NaT
1,EMP0160,99999-01-01,NaT
2,EMP0175,-999,NaT
3,EMP0395,99999-01-01,NaT
4,EMP0551,99999-01-01,NaT
5,EMP0600,-999,NaT
6,EMP0855,-999,NaT
7,EMP0973,-999,NaT


### Issue 4 — `years_experience` out of range

5 rows outside [0, 50]. Replaced with the median for that employee's level.

In [12]:
yrs = pd.to_numeric(raw_employees["years_experience"], errors="coerce")
out_of_range = (yrs < 0) | (yrs > 50)
print("rows affected:", out_of_range.sum())

before = raw_employees.loc[out_of_range, ["employee_id", "level", "years_experience"]]
after = employees_clean.loc[out_of_range, ["employee_id", "level", "years_experience"]]
before.merge(after, on=["employee_id", "level"], suffixes=("_before", "_after"))

rows affected: 5


,employee_id,level,years_experience_before,years_experience_after
0,EMP0216,Junior,-1,1.0
1,EMP0356,Junior,-1,1.0
2,EMP0445,Senior,-1,9.0
3,EMP0583,Senior,-1,9.0
4,EMP0720,Mid,-1,4.0


In [13]:
#full dataframe with the p95 and p05 values for each level
# (same per-level quantiles clean_employees() uses for Issue 5)
salary_num = pd.to_numeric(raw_employees["salary"], errors="coerce")
p95_by_level = salary_num.groupby(raw_employees["level"]).transform(lambda s: s.quantile(0.95))
p05_by_level = salary_num.groupby(raw_employees["level"]).transform(lambda s: s.quantile(0.05))

raw_employees.assign(
    p95_for_level=p95_by_level,
    p05_for_level=p05_by_level,
)

,employee_id,full_name,email,department,role,level,hire_date,salary,manager_id,region,status,years_experience,p95_for_level,p05_for_level
0,EMP0001,Anjali Desai,anjali.desai@presight.ai,Operations,Senior Data Engineer,Mid,2019-08-06,17839,EMP0000,Abu Dhabi,Active,6,23471.25,17396.0
1,EMP0002,Mohammed Mansour,mohammed.mansour@presight.ai,Operations,Senior Data Engineer,Senior,2014-08-20,33928,EMP0000,Dubai,Active,11,35485.30,25898.4
2,EMP0003,Adam Rahman,adam.rahman@presight.ai,Legal,Compliance Analyst,Mid,2021-02-17,18763,EMP0000,Abu Dhabi,Active,4,23471.25,17396.0
3,EMP0004,Fatima Al Suwaidi,fatima.al.suwaidi@presight.ai,Procurement,Procurement Analyst,Mid,2024-05-30,20100,EMP0000,Dubai,Active,2,23471.25,17396.0
4,EMP0005,Priya Chen,priya.chen@presight.ai,Procurement,Procurement Analyst,Junior,2024-02-21,14370,EMP0000,Abu Dhabi,Active,0,16778.40,12238.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,EMP0996,Mariam Al Khaja,mariam.al.khaja@presight.ai,IT,Network Engineer,Mid,2024-09-15,21985,EMP0028,Abu Dhabi,Active,3,23471.25,17396.0
996,EMP0997,Anjali Kapoor,anjali.kapoor2@presight.ai,Finance,Financial Analyst,Senior,2015-02-16,30594,EMP0038,Sharjah,Active,12,35485.30,25898.4
997,EMP0998,Carlos Johnson,carlos.johnson1@presight.ai,Data Science,BI Engineer,Junior,2024-04-21,12164,EMP0042,Abu Dhabi,Active,1,16778.40,12238.5
998,EMP0999,Min Kapoor,min.kapoor@presight.ai,Sales,Sales Analyst,Senior,2016-07-15,27174,EMP0047,Abu Dhabi,Active,9,35485.30,25898.4


### Issue 5 — Salary far outside the level's normal band

3 rows. Capped (winsorised) to the level's 95th percentile instead of dropped.

In [14]:
flagged = employees_clean["salary_flagged_outlier"]
print("rows affected:", flagged.sum())

before = raw_employees.loc[flagged, ["employee_id", "level", "salary"]]
after = employees_clean.loc[flagged, ["employee_id", "level", "salary"]]
before.merge(after, on=["employee_id", "level"], suffixes=("_before", "_after"))

rows affected: 3


,employee_id,level,salary_before,salary_after
0,EMP0356,Junior,76507,16778.4
1,EMP0867,Junior,66797,16778.4
2,EMP0900,Junior,71212,16778.4


### Issue 6 — Active employee under an Inactive manager

A guardrail check — finds 0 rows on this dataset, kept in anyway.

In [15]:
employees_clean.attrs["quality_summary"]["status_conflicts_found"]

0

## Recap

- `projects_clean` — 500 rows, 17 columns (11 raw + 6 derived)
- `employees_clean` — 1,000 rows, 13 columns (12 raw + 1 derived: `salary_flagged_outlier`)
- Every fix above was found by checking the *entire* column, not the first few rows — none of these 6 issues show up if you only look at the first 40 rows of `employees.csv`.